<h1 style="text-align: center;">HEISENBERG-BASED FAULT LOCALIZATION (HBFL)</h1>

<h2 style="text-align: center;">Download Initial Dependencies</h2>

In [15]:
import numpy as np
import pandas as pd
import json
import math
from tqdm import tqdm
from qiskit.quantum_info import SparsePauliOp, Operator, Pauli, Clifford
import qiskit.qasm2 as q2
from qiskit import QuantumCircuit
from IPython.display import display

from heisenbergUtilities import *

In [16]:
# from qiskit_aer import AerSimulator
# simulator = AerSimulator()

# print(simulator.available_devices())

<h2 style="text-align: center;">CONTROL PANEL</h2>

In [17]:
#----------------------------------------------------------------------------------------------------------------------------------------------------------------------#
#-------------------------------------------------------------------------GENERAL PARAMETERS---------------------------------------------------------------------------#
#----------------------------------------------------------------------------------------------------------------------------------------------------------------------#
ALG_NAME = "qft"
ORIGINAL_QASM = f"{ALG_NAME}_5_qubits.qasm"                                                 #| The original algorithm QASM circuit file name.                           
ORIGINAL_QASM_FOLDER = "SBFL_circuits/MQTBench/"                                            #| The folder containing the original QASM circuit file.                    
QASM_MUTANT_FOLDER = f"SBFL_circuits/QMutBenchMutants/Mutants_{ALG_NAME}_5_qubits/"         #| The folder containing all mutants related to the original QASM circuit.  
#----------------------------------------------------------------------------------------------------------------------------------------------------------------------#
#-------------------------------------------------------------------------SB-QOPS PARAMETERS---------------------------------------------------------------------------#
#----------------------------------------------------------------------------------------------------------------------------------------------------------------------#
RUNS = 10                                                                                   #| Number of times SB-QOPS will run to find a failing or passing test case.            
SHOTS = 20000                                                                               #| Number of shots for SB-QOPS to use to create the compact program specification.     
EPOCH = 80                                                                                  #| Number of epochs SB-QOPS will run to navigate the search space of test cases.       
SBQOPS_TOL = 0.1                                                                            #| Tolerance value SB-QOPS uses to determine if a test case is failing or passing.     
#----------------------------------------------------------------------------------------------------------------------------------------------------------------------#
#---------------------------------------------------------------------HEISENBERG SBFL PARAMETERS-----------------------------------------------------------------------#
#----------------------------------------------------------------------------------------------------------------------------------------------------------------------#
LAMBDA_PHASE = 2                                                                        #| The hyperparameter used to weight the phase change of the Pauli propagation
LAMBDA_CHANGE = 1                                                                         #| The hyperparameter used to weight the change of string of the Pauli propagation
LAMBDA_SIMILARITY = 1                                                                       #| The hyperparameter used to weight the similarity difference between current and target Pauli
ATOL = 1e-4                                                                                 #| The tolerance value used to prune negligible magnitudes during Pauli propagation.   
MAX_TERMS = 100                                                                             #| The maximum number of terms to keep during Pauli propagation. If None, use all.     
SEARCH_STEP = 3                                                                             #| The search step size used during Pauli propagation. If None, pauli-prop uses 4.     
VERBOSE = 1                                                                                 #| A boolean value to determine if the program should print out detailed information.  
#----------------------------------------------------------------------------------------------------------------------------------------------------------------------#


Generate the linked list of all operations that take place in the inverse circuit

In [18]:
"""
LinkedListNode: A class that holds information related to a gate in a quantum circuit. 

PROPERTIES:
    - value (String): The name of the quantum gate
    - depth (int): The depth of the gate in the circuit
    - count (Int): The probability that the gate will meaningfully change the Pauli string.
    - next (LinkedListNode): The next gate in the circuit

"""

class LinkedListNode:
    def __init__(self, name="Initial", depth=0, count=0,  next=None):
        self.value = name
        self.depth = depth
        self.next = next
        self.count = count

class LinkedList:
    def __init__(self):
        self.head = None

    def append(self, name, depth):
        new_node = LinkedListNode(name, depth)
        if not self.head:
            self.head = new_node
            return
        last_node = self.head
        while last_node.next:
            last_node = last_node.next
        last_node.next = new_node

    def mark(self, depth):
        current_node = self.head
        while current_node:
            if current_node.depth == depth:
                current_node.count += 1
                return
            current_node = current_node.next

    def reset(self):
        current_node = self.head
        while current_node:
            current_node.count = 0
            current_node = current_node.next

<h3>Download MQT Benchmark circuits.</h3>

We'll start with DJ, RealAmpRandom, and GHZ since SB-QOPS caught those mutants 100% of the time for 5 qubit circuits



<h3>Generate mutants.</h3>

We'll start with mutants that add an unnecessary gate, then add mutants that replace a certain gate later.

The mutants were downloaded from QMutBench, choosing specifically mutants that added a gate anywhere with 0-10% survival scores

In [19]:
correct_circuit = load_program(ORIGINAL_QASM, ORIGINAL_QASM_FOLDER).copy()
correct_list = LinkedList()
correct_list = construct_list(correct_list, correct_circuit, inverse=False)

<h2 style="text-align: center;"> SB-QOPS </h2>

This is where we will use SB-QOPS on a provided circuit and its mutants

For a single mutant, it took about 2 minutes to generate a 20 test suite (10 fail, 10 pass) on an RTX 4090 SUPER. It can only be run on a Linux environment since the AER Simulator does not support Windows

There are 113 mutants for the DJ algorithm in use: 226 minutes or 3 3/4 hours

There are 89 mutants for the GHZ algorithm in use: 178 minutes or about 3 hours

There are 125 mutants for the RealAmpRandom algorithm in use: 250 minutes or just over 4 hours

BUT this cell should only need to be run once per algorithm with mutants under test. After it saves to the csv file at the end, this cell can be commented out as to not accidentally run it again. 

In the future if we want to test this methodology on higher qubit circuits or other algorithms, we'll likely either want to reduce the number of tests in the suite or the number of mutants we're analyzing

In [20]:
# import QOPS as qops
# from QOPS_test import get_compact_program_specification_Z
# from pathlib import Path

# ga_result = pd.DataFrame(columns=['Program',"path_to_mutant",'catch_avg','avg_fitness','failing_testcases', 'passing_testcases'])
# program_history = {}

# #program_specification = The compact program specification. SB-QOPS needs this to generate failing test cases.
# # In a public-use environment, this would be provided by the user.
# program_specification = get_compact_program_specification_Z(correct_circuit, shots=SHOTS)

# #mutants = A python list of qiskit QuantumCircuit variables with a mutation in them
# mutants = []
# mutants_names = []
# root = Path(QASM_MUTANT_FOLDER)
# for qasm_mutant in root.rglob("*.qasm"):
#     mutants.append(load_program(qasm_mutant.name, QASM_MUTANT_FOLDER))
#     mutants_names.append(qasm_mutant.name)

# for mutant_index,mutant in enumerate(mutants):
#     tester = qops.Circuit_Tester(CUT=mutant)
#     tester.set_applicable_families_Z(program_specification)
#     mutants_per_run = []
#     fitness_per_run = []
#     failing_testcases_per_run = []
#     history_per_run = []

#     print("NOW RUNNING TO FIND FAILING TESTS")
#     #For a predefined number of attempts, try to find a set of failing test cases (output is above predefined tolerance)
#     for runs in range(RUNS):
#         killed = 0
#         pauli = {}
#         fitness = 0
#         for i in range(len(tester.applicable_families)):
#             best_function,testcase, history = tester.run_mealoneplusone(i, EPOCH)
#             if best_function > SBQOPS_TOL:
#                 killed = 1
#                 pauli = testcase
#                 fitness = best_function
#                 break
#         mutants_per_run.append(killed)
#         failing_testcases_per_run.append(pauli)
#         fitness_per_run.append(fitness)
#         history_per_run.append(history)

#     avg_score = np.mean(mutants_per_run)
#     avg_fitness = np.mean(fitness_per_run)

#     #Here is our new, modified algorithm from the SB-QOPS method that attempts to find passing test cases as well. We'll need them for SBFL
#     passing_testcases_per_run = []

#     print("NOW RUNNING TO FIND PASSING TESTS")
#     for runs in range(RUNS):
#         pauli = {}
#         fitness = 0
#         for i in range(len(tester.applicable_families)):
#             best_function, testcase, history = tester.run_reverse_mealoneplusone(i,EPOCH)
#             if best_function < SBQOPS_TOL:
#                 pauli = testcase
#                 break
#         passing_testcases_per_run.append(pauli)

#     #Replace static program file with "program_name" when ready to test fully
#     """
#     ga_result: A pandas dataframe with the following columns
#         Program: Name of the mutant quantum circuit file
#         Path_To_Mutant: Path to the mutant file
#         Catch_Avg: The average number of times the mutant was identified by SB-QOPS
#         Avg_Fitness: The average fitness of the search algorithm used. Higher usually indicates higher Catch_Avg
#         Failing_Testcases: A list of dicts indicating the set of Pauli strings and the weights that should be applied to generate a failing test case
#         Passing_Testcases: A list of dicts indicating the set of Pauli strings and the weights that should be applied to generate a passing test case
#     """
#     ga_result = pd.concat([pd.DataFrame([[mutants_names[mutant_index],QASM_MUTANT_FOLDER,avg_score,avg_fitness,json.dumps(failing_testcases_per_run), json.dumps(passing_testcases_per_run)]],columns=ga_result.columns),ga_result],ignore_index=True)
#     ga_result.to_csv(f'realamprandom_ga_result', index=False)

ga_result is a slightly altered pandas Dataframe with similar structure found in SB-QOPS's results folder. The differences are changing the column "Program" to be the name of the mutant file instead of the original quantum circuit, changing "Mutant" to be the path to the mutant file instead of being an arbitrary index value, and adding a passing_testcases column that returns Pauli strings and coefficients that generate passing tests.

Now we want to run SBFL using Heisenberg evolution trees on each circuit placed in ga_result

Evolution analysis tends to take about 5 minutes for 10 failing tests on a very complex algorithm such as QNN, so about 10 minutes for 20 total tests

DJ was a relatively small algorithm with few to no branching gates, so it only ended up taking about 5-6 minutes to evolve and analyze all 113 DJ mutants

About 890 minutes for GHZ, or 14.8 hours to evolve and analyze all 89 GHZ mutants

About 1250 minutes for RealAmpRandom, or 20.8 hours to evolve and analyze all 125 RealAmpRandom mutants

The runtime of this code segment is directly tied to the depth of the circuit being analyzed and the number of 2-branching gates such as rotation or phase gates that exist in the circuit.

This is something to note in the final paper. Regardless of accuracy this methodology takes a long time to run. If results are promising, then we'll want to look into the tradeoffs between EXAM and hyperparameters to speed this up. Primarily the atol, max_terms, and search_step parameters in the evolve_pauli_exact method in HeisenbergUtilities.py


<h2 style="text-align: center;">HEISENBERG EVOLUTIONS (PAULI PROPAGATION)</h2>

In [21]:
from heisenbergTree import *

ga_result = pd.read_csv(f'{ALG_NAME}_ga_result.csv')
tarantula_sbfl_result = pd.DataFrame(columns=['Program','path_to_mutant','exam_score'])
barinel_sbfl_result = pd.DataFrame(columns=['Program','path_to_mutant','exam_score'])
dstar_sbfl_result = pd.DataFrame(columns=['Program','path_to_mutant','exam_score'])
mutant_num = 0

#For each mutant of a circuit, run the SBFL implementation
for mutant, path in zip(ga_result.loc[:,"Program"], ga_result.loc[:,"path_to_mutant"]):
    print("Now evolving the following mutant: ", mutant)

    #Extract the raw test cases found for each mutant
    desired_failing_testcases = ga_result.loc[(ga_result["Program"] == mutant), ["failing_testcases"]]
    desired_passing_testcases = ga_result.loc[(ga_result["Program"] == mutant), ["passing_testcases"]]

    dict_failing_testcases = desired_failing_testcases.to_dict()
    dict_passing_testcases = desired_passing_testcases.to_dict()

    string_failing_testcases = dict_failing_testcases['failing_testcases'][mutant_num]
    string_passing_testcases = dict_passing_testcases['passing_testcases'][mutant_num]

    list_failing_testcases = json.loads(string_failing_testcases)
    list_passing_testcases = json.loads(string_passing_testcases)

    #Remove empty tests
    raw_failing_testcases = remove_null_tests(list_failing_testcases)
    raw_passing_testcases = remove_null_tests(list_passing_testcases)

    circuit_inverse = load_program(mutant,path).copy().inverse()  # Invert to track backward evolution of the operator
    forward_mutant = load_program(mutant, path).copy()

    #Create the linked list of operations in the inverse circuit
    operation_list = LinkedList()
    operation_list = construct_list(operation_list, circuit_inverse, True)

    forward_list = LinkedList()
    forward_list = construct_list(forward_list, forward_mutant, False)

    #Create list of nodes in linked list. Somewhere down the road remove the linked list implementation. It's unnecessary and giving me a headache
    node_list = []
    node = operation_list.head
    while node:
        node_list.append(node)
        node = node.next

    #Perform Pauli propagation (Heisenberg evolution) for all test cases. Returns a dataframe with all counts for all operations
    global_frame_fail = heisenberg_evolve(circuit_inverse, 
                                          operation_list, 
                                          raw_failing_testcases, 
                                          "fail", 
                                          LAMBDA_PHASE, 
                                          LAMBDA_CHANGE, 
                                          LAMBDA_SIMILARITY, 
                                          atol = ATOL, 
                                          max_terms = MAX_TERMS, 
                                          search_step = SEARCH_STEP)
    
    global_frame_pass = heisenberg_evolve(circuit_inverse, 
                                          operation_list, 
                                          raw_passing_testcases, 
                                          "pass", 
                                          LAMBDA_PHASE, 
                                          LAMBDA_CHANGE, 
                                          LAMBDA_SIMILARITY, 
                                          atol = ATOL, 
                                          max_terms = MAX_TERMS, 
                                          search_step = SEARCH_STEP)

    global_frame = pd.concat([global_frame_fail, global_frame_pass], ignore_index=True)

    global_frame = global_frame.drop(["Initial"],axis=1)
    if VERBOSE:
        display(global_frame)

    tarantula_scores = tarantula(global_frame)
    barinel_scores = barinel(global_frame)
    dstar_scores = dstar(global_frame)

    if VERBOSE:
        print("BARINEL SCORES")
        display(barinel_scores)
        print("TARANTULA SCORES")
        display(tarantula_scores)
        print("DSTAR SCORES")
        display(dstar_scores)
    
    # Here is where analysis of the methodology begins. 
    # We first extract the position of the erroneous gate by comparing it to the original MQT gate
    # NOTE: This method is intended for single-added gates for now. Other types of mutants will require other methods later
    error_gate_location = find_erroneous_gate(forward_mutant, correct_circuit)

    if VERBOSE:
        print("ERROR GATE LOCATION:")
        print(error_gate_location)

    # Calculate the EXAM score for Barinel by counting the number of gates we would have to observe before we reach the erroneous gate.
    gates_observed = 0
    barinel_exam_score = 0
    for col_name, col_date in barinel_scores.items():
        gates_observed += 1

        #Extract depth from column name
        gate_depth = int(col_name.split()[1])

        if gate_depth == error_gate_location:
            barinel_exam_score = gates_observed/(circuit_inverse.size())
            break

    # Calculate the EXAM score for Tarantula by counting the number of gates we would have to observe before we reach the erroneous gate.
    gates_observed = 0
    tarantula_exam_score = 0
    for col_name, col_date in tarantula_scores.items():
        gates_observed += 1

        #Extract depth from column name
        gate_depth = int(col_name.split()[1])

        if gate_depth == error_gate_location:
            tarantula_exam_score = gates_observed/(circuit_inverse.size())
            break

    # Calculate the EXAM score for DStar by counting the number of gates we would have to observe before we reach the erroneous gate.
    gates_observed = 0
    dstar_exam_score = 0
    for col_name, col_date in dstar_scores.items():
        gates_observed += 1

        #Extract depth from column name
        gate_depth = int(col_name.split()[1])

        if gate_depth == error_gate_location:
            dstar_exam_score = gates_observed/(circuit_inverse.size())
            break

    # Here we store the results from the HBFL analysis for both barinel and tarantula into a csv file for later analysis.
    barinel_sbfl_result = pd.concat([pd.DataFrame([[mutant,QASM_MUTANT_FOLDER,barinel_exam_score]],columns=barinel_sbfl_result.columns),barinel_sbfl_result],ignore_index=True)
    barinel_sbfl_result.sort_values(by=['exam_score'], ascending=False)
    barinel_sbfl_result.to_csv(f'{ALG_NAME}_barinel_sbfl_result.csv', index=False)

    tarantula_sbfl_result = pd.concat([pd.DataFrame([[mutant,QASM_MUTANT_FOLDER,tarantula_exam_score]],columns=tarantula_sbfl_result.columns),tarantula_sbfl_result],ignore_index=True)
    tarantula_sbfl_result.sort_values(by=['exam_score'], ascending=False)
    tarantula_sbfl_result.to_csv(f'{ALG_NAME}_tarantula_sbfl_result.csv', index=False)

    dstar_sbfl_result = pd.concat([pd.DataFrame([[mutant,QASM_MUTANT_FOLDER,dstar_exam_score]],columns=dstar_sbfl_result.columns),dstar_sbfl_result],ignore_index=True)
    dstar_sbfl_result.sort_values(by=['exam_score'], ascending=False)
    dstar_sbfl_result.to_csv(f'{ALG_NAME}_dstar_sbfl_result.csv', index=False)

    mutant_num += 1

if VERBOSE:
    display(barinel_sbfl_result)
    display(tarantula_sbfl_result)
    

Now evolving the following mutant:  AddGate_swap_inGap_2_.qasm


100%|██████████| 10/10 [00:28<00:00,  2.85s/it]


,swap 17,swap 16,h 15,cp 14,cp 13,cp 12,cp 11,h 10,cp 9,cp 8,cp 7,h 6,cp 5,cp 4,h 3,swap 2,cp 1,h 0,pass/fail
0,0.393206,0.181854,0.248567,0.348279,0.118600,0.036331,0.012451,0.251162,0.323315,0.114945,0.037600,0.275747,0.350704,0.129532,0.322153,0.189299,0.407668,0.204961,fail
1,0.615940,0.322683,0.325194,0.491316,0.170919,0.051847,0.016656,0.355000,0.506835,0.164590,0.060238,0.366473,0.476551,0.181669,0.401399,0.270595,0.514582,0.257597,fail
2,0.642780,0.325768,0.377003,0.571659,0.193524,0.061905,0.017545,0.373172,0.477899,0.165902,0.054448,0.459401,0.577969,0.204169,0.475172,0.280655,0.545148,0.290363,fail
3,0.534969,0.343645,0.367399,0.571376,0.188290,0.065172,0.017823,0.399148,0.530893,0.195922,0.064500,0.422147,0.550027,0.189684,0.449587,0.281856,0.540144,0.277630,fail
4,0.616170,0.285635,0.254444,0.368546,0.136406,0.035294,0.012629,0.295494,0.396582,0.128379,0.044883,0.384137,0.466618,0.170183,0.348359,0.235903,0.422108,0.221783,fail
5,0.512517,0.276398,0.360068,0.539790,0.183813,0.059220,0.016964,0.396784,0.506067,0.177463,0.061126,0.440843,0.520877,0.184964,0.508357,0.238097,0.579278,0.316582,fail
6,0.567234,0.328166,0.370217,0.567947,0.188535,0.060416,0.018191,0.381126,0.487729,0.173744,0.059544,0.438053,0.532498,0.188935,0.440029,0.254755,0.512599,0.275375,fail
7,0.610808,0.287032,0.347996,0.532385,0.174686,0.055031,0.017026,0.400215,0.529095,0.177455,0.063732,0.381271,0.474646,0.173799,0.415325,0.268025,0.510209,0.265845,fail
8,0.582942,0.311960,0.330539,0.511246,0.168902,0.054658,0.016332,0.387104,0.532122,0.181227,0.063354,0.374155,0.484328,0.182087,0.431539,0.274910,0.525842,0.273019,fail
9,0.730636,0.383497,0.537270,0.835564,0.304057,0.090632,0.027054,0.509326,0.682492,0.229797,0.080442,0.575726,0.698496,0.249645,0.571013,0.351633,0.672973,0.355128,fail


BARINEL SCORES


,swap 16,swap 17,cp 12,cp 14,cp 11,h 15,swap 2,cp 1,cp 13,cp 7,h 10,h 3,h 0,cp 9,cp 8,cp 5,h 6,cp 4
sum,0.586517,0.584319,0.542927,0.540654,0.539586,0.535508,0.534764,0.534381,0.529243,0.522954,0.52275,0.52247,0.520843,0.52071,0.519997,0.506729,0.50391,0.500996


...

DSTAR SCORES


,cp 1,cp 8,cp 4,cp 13,h 5,h 9,h 2,swap 17,h 16,h 14,h 0,swap 15,cp 3,cp 7,cp 12,cp 6,cp 11,cp 10
sum,2.960015,2.189731,2.134115,1.918817,1.792639,1.657491,1.636929,1.491379,1.417121,1.346198,1.24786,0.494305,0.48204,0.336087,0.314478,0.051994,0.031963,0.004553


ERROR GATE LOCATION:
15


,Program,path_to_mutant,exam_score
0,AddGate_h_inGap_6_.qasm,SBFL_circuits/QMutBenchMutants/Mutants_qft_5_q...,0.777778
1,AddGate_h_inGap_9_.qasm,SBFL_circuits/QMutBenchMutants/Mutants_qft_5_q...,0.055556
2,AddGate_cswap_inGap_3_.qasm,SBFL_circuits/QMutBenchMutants/Mutants_qft_5_q...,0.277778
3,AddGate_swap_inGap_3_.qasm,SBFL_circuits/QMutBenchMutants/Mutants_qft_5_q...,0.111111
4,AddGate_cswap_inGap_4_.qasm,SBFL_circuits/QMutBenchMutants/Mutants_qft_5_q...,0.611111
5,AddGate_ry_inGap_4_.qasm,SBFL_circuits/QMutBenchMutants/Mutants_qft_5_q...,0.166667
6,AddGate_swap_inGap_4_.qasm,SBFL_circuits/QMutBenchMutants/Mutants_qft_5_q...,0.888889
7,AddGate_cswap_inGap_5_.qasm,SBFL_circuits/QMutBenchMutants/Mutants_qft_5_q...,0.833333
8,AddGate_h_inGap_12_.qasm,SBFL_circuits/QMutBenchMutants/Mutants_qft_5_q...,0.055556
9,AddGate_h_inGap_7_.qasm,SBFL_circuits/QMutBenchMutants/Mutants_qft_5_q...,0.055556


,Program,path_to_mutant,exam_score
0,AddGate_h_inGap_6_.qasm,SBFL_circuits/QMutBenchMutants/Mutants_qft_5_q...,0.777778
1,AddGate_h_inGap_9_.qasm,SBFL_circuits/QMutBenchMutants/Mutants_qft_5_q...,0.055556
2,AddGate_cswap_inGap_3_.qasm,SBFL_circuits/QMutBenchMutants/Mutants_qft_5_q...,0.277778
3,AddGate_swap_inGap_3_.qasm,SBFL_circuits/QMutBenchMutants/Mutants_qft_5_q...,0.111111
4,AddGate_cswap_inGap_4_.qasm,SBFL_circuits/QMutBenchMutants/Mutants_qft_5_q...,0.611111
5,AddGate_ry_inGap_4_.qasm,SBFL_circuits/QMutBenchMutants/Mutants_qft_5_q...,0.166667
6,AddGate_swap_inGap_4_.qasm,SBFL_circuits/QMutBenchMutants/Mutants_qft_5_q...,0.888889
7,AddGate_cswap_inGap_5_.qasm,SBFL_circuits/QMutBenchMutants/Mutants_qft_5_q...,0.833333
8,AddGate_h_inGap_12_.qasm,SBFL_circuits/QMutBenchMutants/Mutants_qft_5_q...,0.055556
9,AddGate_h_inGap_7_.qasm,SBFL_circuits/QMutBenchMutants/Mutants_qft_5_q...,0.055556
